In [1]:
print("Hello")

Hello


In [1]:
import joblib
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("Churn_Modelling.csv")
df.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


In [5]:
df.shape

(10000, 14)

In [6]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [3]:
TARGET_COLUMN = 'Exited'
ID_COLUMNS = ['RowNumber', 'CustomerId', 'Surname']
RANDOM_STATE = 42

In [4]:
X = df.drop(columns=[TARGET_COLUMN] + ID_COLUMNS)
y = df[TARGET_COLUMN]

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

/var/folders/k8/kc8jrng519q81_pvhp9qvnyr0000gn/T/ipykernel_6557/1568736471.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


In [42]:
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Categorical features: ['Geography', 'Gender']


In [ ]:
X_reference, X_current, y_reference, y_current = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Reference data shape:', X_reference.shape)
print('Current data shape:', X_current.shape)

Reference data shape: (7000, 10)
Current data shape: (3000, 10)


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

In [46]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier( random_state=RANDOM_STATE))
])

In [47]:
model.fit(X_reference, y_reference)

reference_predictions = model.predict(X_reference)
current_predictions = model.predict(X_current)

reference_probabilities = model.predict_proba(X_reference)[:, 1]
current_probabilities = model.predict_proba(X_current)[:, 1]

In [48]:
print('Reference Accuracy:', round(accuracy_score(y_reference, reference_predictions), 3))
print('Current Accuracy:', round(accuracy_score(y_current, current_predictions), 3))
print('Reference ROC AUC:', round(roc_auc_score(y_reference, reference_probabilities), 3))
print('Current ROC AUC:', round(roc_auc_score(y_current, current_probabilities), 3))

Reference Accuracy: 1.0
Current Accuracy: 0.864
Reference ROC AUC: 1.0
Current ROC AUC: 0.862


In [49]:
print('Classification report on current data:')
print(classification_report(y_current, current_predictions))

Classification report on current data:
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      2389
           1       0.78      0.46      0.58       611

    accuracy                           0.86      3000
   macro avg       0.83      0.72      0.75      3000
weighted avg       0.86      0.86      0.85      3000



In [50]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
))
])

In [51]:
model.fit(X_reference, y_reference)

reference_predictions = model.predict(X_reference)
current_predictions = model.predict(X_current)

reference_probabilities = model.predict_proba(X_reference)[:, 1]
current_probabilities = model.predict_proba(X_current)[:, 1]

In [52]:
print('Reference Accuracy:', round(accuracy_score(y_reference, reference_predictions), 3))
print('Current Accuracy:', round(accuracy_score(y_current, current_predictions), 3))
print('Reference ROC AUC:', round(roc_auc_score(y_reference, reference_probabilities), 3))
print('Current ROC AUC:', round(roc_auc_score(y_current, current_probabilities), 3))

Reference Accuracy: 0.978
Current Accuracy: 0.854
Reference ROC AUC: 0.997
Current ROC AUC: 0.869


In [53]:
print('Classification report on current data:')
print(classification_report(y_current, current_predictions))

Classification report on current data:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      2389
           1       0.67      0.55      0.61       611

    accuracy                           0.85      3000
   macro avg       0.78      0.74      0.76      3000
weighted avg       0.85      0.85      0.85      3000



In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight="balanced",random_state=RANDOM_STATE))
])

In [ ]:
model.fit(X_reference, y_reference)

reference_predictions = model.predict(X_reference)
current_predictions = model.predict(X_current)

reference_probabilities = model.predict_proba(X_reference)[:, 1]
current_probabilities = model.predict_proba(X_current)[:, 1]

In [ ]:
print('Reference Accuracy:', round(accuracy_score(y_reference, reference_predictions), 3))
print('Current Accuracy:', round(accuracy_score(y_current, current_predictions), 3))
print('Reference ROC AUC:', round(roc_auc_score(y_reference, reference_probabilities), 3))
print('Current ROC AUC:', round(roc_auc_score(y_current, current_probabilities), 3))

Reference Accuracy: 0.703
Current Accuracy: 0.718
Reference ROC AUC: 0.76
Current ROC AUC: 0.792


In [ ]:
print(classification_report(y_current, current_predictions))

              precision    recall  f1-score   support

           0       0.91      0.72      0.80      2389
           1       0.40      0.73      0.51       611

    accuracy                           0.72      3000
   macro avg       0.65      0.72      0.66      3000
weighted avg       0.81      0.72      0.74      3000



In [58]:
print('Confusion matrix:')
print(confusion_matrix(y_current, current_predictions))

Confusion matrix:
[[1711  678]
 [ 167  444]]


In [ ]:
production_df = X_current.copy()

In [ ]:
#numeric
production_df['Age'] = production_df['Age'] + 5
production_df['Balance'] = production_df['Balance'] * 1.15
production_df['EstimatedSalary'] = production_df['EstimatedSalary'] * 0.90

In [ ]:
#categorical
sample_index = production_df.sample(frac=0.20, random_state=RANDOM_STATE).index
production_df.loc[sample_index, 'Geography'] = 'Germany'

production_df.head(10)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
6417,790,Germany,Male,47,6,0.000000,2,1,1,96782.0481
199,521,France,Male,45,6,127520.528400,1,1,0,8495.6364
2051,712,France,Female,47,1,141350.783750,2,0,0,137203.3161
8481,729,Spain,Female,48,10,0.000000,2,1,0,153678.9672
1071,695,Germany,Male,62,8,136248.261350,1,1,1,18213.3684
6533,489,France,Female,57,8,137400.317550,2,1,1,87176.6226
6397,550,Spain,Female,43,9,96262.169675,1,1,1,83922.5286
9018,663,Germany,Male,77,9,0.000000,3,1,1,58578.2037
3266,537,Spain,Male,40,1,136400.229825,1,1,1,78209.8902
2681,673,Germany,Male,46,5,96658.959350,2,0,0,158875.2306


In [ ]:
def calculate_psi(reference_values, current_values, bins=10):
    reference_values = pd.Series(reference_values).dropna()
    current_values = pd.Series(current_values).dropna()

    _, bin_edges = np.histogram(reference_values, bins=bins)
    reference_counts, _ = np.histogram(reference_values, bins=bin_edges)
    current_counts, _ = np.histogram(current_values, bins=bin_edges)

    reference_percent = reference_counts / max(reference_counts.sum(), 1)
    current_percent = current_counts / max(current_counts.sum(), 1)

    reference_percent = np.where(reference_percent == 0, 0.0001, reference_percent)
    current_percent = np.where(current_percent == 0, 0.0001, current_percent)

    psi_values = (current_percent - reference_percent) * np.log(current_percent / reference_percent)
    return float(np.sum(psi_values))

In [ ]:
def psi_status(psi):
    if psi >= 0.25:
        return 'Strong drift'
    if psi >= 0.10:
        return 'Warning'
    return 'No major drift'


def p_value_status(p_value):
    if p_value < 0.05:
        return 'Possible drift'
    return 'No major drift'

In [ ]:
numeric_drift_rows = []

for column in numeric_features:
    ks_statistic, ks_p_value = ks_2samp(X_reference[column], production_df[column])
    psi = calculate_psi(X_reference[column], production_df[column])

    numeric_drift_rows.append({
        'feature': column,
        'type': 'numeric',
        'reference_mean': X_reference[column].mean(),
        'current_mean': production_df[column].mean(),
        'ks_statistic': ks_statistic,
        'ks_p_value': ks_p_value,
        'psi': psi,
        'psi_status': psi_status(psi),
        'ks_status': p_value_status(ks_p_value)
    })

numeric_drift_report = pd.DataFrame(numeric_drift_rows)
display(numeric_drift_report.sort_values('psi', ascending=False))

,feature,type,reference_mean,current_mean,ks_statistic,ks_p_value,psi,psi_status,ks_status
1,Age,numeric,38.998429,48.743000,0.440810,6.694590e-321,1.347627,Strong drift,Possible drift
7,EstimatedSalary,numeric,99466.316050,82252.310344,0.186190,4.294847e-64,0.839804,Strong drift,Possible drift
3,Balance,numeric,76314.462261,101681.583816,0.285143,5.503762e-151,0.725550,Strong drift,Possible drift
2,Tenure,numeric,5.010571,5.018000,0.013238,8.507245e-01,0.005462,No major drift,No major drift
0,CreditScore,numeric,650.562429,650.450333,0.010143,9.807561e-01,0.003188,No major drift,No major drift
4,NumOfProducts,numeric,1.533000,1.523667,0.004667,1.000000e+00,0.000616,No major drift,No major drift
5,HasCrCard,numeric,0.707429,0.701000,0.006429,9.999923e-01,0.000198,No major drift,No major drift
6,IsActiveMember,numeric,0.517000,0.510667,0.006333,9.999948e-01,0.000161,No major drift,No major drift


In [ ]:
categorical_drift_rows = []

for column in categorical_features:
    reference_counts = X_reference[column].value_counts()
    current_counts = production_df[column].value_counts()
    all_categories = sorted(set(reference_counts.index).union(set(current_counts.index)))

    table = pd.DataFrame({
        'reference': [reference_counts.get(category, 0) for category in all_categories],
        'current': [current_counts.get(category, 0) for category in all_categories]
    }, index=all_categories)

    chi2, p_value, _, _ = chi2_contingency(table.T)

    categorical_drift_rows.append({
        'feature': column,
        'type': 'categorical',
        'chi2_statistic': chi2,
        'chi2_p_value': p_value,
        'status': p_value_status(p_value)
    })

categorical_drift_report = pd.DataFrame(categorical_drift_rows)
display(categorical_drift_report)

,feature,type,chi2_statistic,chi2_p_value,status
0,Geography,categorical,232.306947,3.590769e-51,Possible drift
1,Gender,categorical,3.994052,4.566114e-02,Possible drift


In [71]:
len(production_df)

3000

In [ ]:
production_probabilities = model.predict_proba(production_df)[:, 1]
production_predictions = model.predict(production_df)


In [ ]:
results = production_df.copy()
results['predicted_exited'] = production_predictions
results['predicted_churn_probability'] = production_probabilities

print(results[['predicted_exited', 'predicted_churn_probability']].head(100))

      predicted_exited  predicted_churn_probability
6417                 0                     0.466963
199                  1                     0.592891
2051                 1                     0.725371
8481                 1                     0.666647
1071                 1                     0.799483
...                ...                          ...
4758                 1                     0.826947
7307                 0                     0.363942
9159                 1                     0.540874
5041                 0                     0.416234
536                  1                     0.761970

[100 rows x 2 columns]


In [ ]:
prediction_ks_statistic, prediction_ks_p_value = ks_2samp(reference_probabilities, production_probabilities)
prediction_psi = calculate_psi(reference_probabilities, production_probabilities)

In [ ]:
prediction_drift_report = pd.DataFrame([{
    'check': 'prediction_probability_drift',
    'reference_average_churn_probability': reference_probabilities.mean(),
    'production_average_churn_probability': production_probabilities.mean(),
    'ks_statistic': prediction_ks_statistic,
    'ks_p_value': prediction_ks_p_value,
    'psi': prediction_psi,
    'psi_status': psi_status(prediction_psi),
    'ks_status': p_value_status(prediction_ks_p_value)
}])

display(prediction_drift_report)

,check,reference_average_churn_probability,production_average_churn_probability,ks_statistic,ks_p_value,psi,psi_status,ks_status
0,prediction_probability_drift,0.440906,0.621472,0.339714,1.739115e-215,0.755033,Strong drift,Possible drift


In [ ]:
display(pd.DataFrame({
    'prediction': production_predictions,
    'probability': production_probabilities
}).head(100))

,prediction,probability
0,0,0.466963
1,1,0.592891
2,1,0.725371
3,1,0.666647
4,1,0.799483
...,...,...
95,1,0.826947
96,0,0.363942
97,1,0.540874
98,0,0.416234


In [ ]:
alerts = []

for _, row in numeric_drift_report.iterrows():
    if row['psi_status'] == 'Strong drift' or row['ks_status'] == 'Possible drift':
        alerts.append(f"Numeric drift alert: {row['feature']} | PSI={row['psi']:.3f} | KS p-value={row['ks_p_value']:.4f}")

for _, row in categorical_drift_report.iterrows():
    if row['status'] == 'Possible Drift':
        alerts.append(f"Categorical drift alert: {row['feature']} | Chi-square p-value={row['chi2_p_value']:.4f}")

if prediction_drift_report.loc[0, 'psi_status'] == 'Strong Drift' or prediction_drift_report.loc[0, 'ks_status'] == 'Possible Drift':
    alerts.append(
        'Prediction drift alert: model churn probabilities changed '
        f"| PSI={prediction_drift_report.loc[0, 'psi']:.3f} "
        f"| KS p-value={prediction_drift_report.loc[0, 'ks_p_value']:.4f}"
    )

if not alerts:
    alerts.append('No major drift detected.')

for alert in alerts:
    print(alert)

Numeric drift alert: Age | PSI=1.348 | KS p-value=0.0000
Numeric drift alert: Balance | PSI=0.726 | KS p-value=0.0000
Numeric drift alert: EstimatedSalary | PSI=0.840 | KS p-value=0.0000


In [82]:
numeric_drift_report.to_csv('numeric_drift_report.csv', index=False)
categorical_drift_report.to_csv('categorical_drift_report.csv', index=False)
prediction_drift_report.to_csv('prediction_drift_report.csv', index=False)


In [83]:
production_predictions_df = production_df.copy()
production_predictions_df['predicted_exited'] = production_predictions
production_predictions_df['predicted_churn_probability'] = production_probabilities
production_predictions_df.to_csv('production_predictions.csv', index=False)

In [84]:
with open('alerts.txt', 'w') as file:
    file.write('\n'.join(alerts))

joblib.dump(model,'churn_model.joblib')

['churn_model.joblib']

In [ ]:
from evidently import Report
from evidently.metrics import ColumnDriftMetric
from IPython.display import HTML, display
import pandas as pd

reference_data = pd.DataFrame({
    "prediction_probability": reference_probabilities
})

current_data = pd.DataFrame({
    "prediction_probability": production_probabilities
})

report = Report(metrics=[
    ColumnDriftMetric(column_name="prediction_probability")
])

report.run(
    reference_data=reference_data,
    current_data=current_data
)

report.save_html("prediction_drift_report.html")

display(HTML(open("prediction_drift_report.html", "r", encoding="utf-8").read()))

KeyError: ~TResult